# Task 1.1 — Core Contribution / Architecture


## Step-by-Step Method Description

This paper addresses **pairwise classification** — the task of deciding whether two inputs belong to the same class. A concrete example is face verification: given two face images, predict same person or different person. Standard SVMs work on single inputs. This paper extends them to work on *pairs* of inputs while preserving a natural symmetry property.

---

### Step 1: Pair Construction

**Description:**  
Two data points $a$ and $b$ (e.g., two face images) are paired to form a single input $(a, b)$ for the classifier. The full training set becomes a collection of such labeled pairs: same class → label +1, different class → label −1.

**Reference:** Section 2, Equation (1) — formal definition of pairwise training data.  

**Purpose:** Converts a multi-class recognition problem into a binary classification problem on pairs, allowing SVM to be applied directly.

---

### Step 2: Applying a Pairwise Kernel

**Description:**  
Instead of a standard kernel $k(x_i, x_j)$ over individual points, a **pairwise kernel** is defined over pairs. The two main options are:

- **Direct Kernel (K_D):**  
  $K_D((a,b),(c,d)) = k(a,c) + k(b,d)$  
  Measures similarity of the first elements and second elements separately, then adds them.

- **Tensor Kernel (K_T):**  
  $K_T((a,b),(c,d)) = k(a,c) \cdot k(b,d)$  
  Measures joint similarity as a product.

**Reference:** Section 2, Equations (4) and (5) — definitions of Direct and Tensor pairwise kernels.

**Purpose:** Allows the SVM to operate in a pair-space kernel, capturing relationships between the two inputs jointly.

---

### Step 3: Enforcing Symmetry via Balanced Kernels

**Description:**  
A critical requirement for pairwise classification is **symmetry**: the decision for $(a,b)$ should be the same as for $(b,a)$ — the order of the two inputs should not matter. A plain Direct or Tensor kernel does not guarantee this. The paper introduces **balanced (linearised) versions** that do:

- **Direct Linearised (K_DL):**  
  $K_{DL}((a,b),(c,d)) = \\frac{1}{2}[k(a,c) + k(b,d) + k(a,d) + k(b,c)]$

- **Tensor Linearised (K_TL):**  
  $K_{TL}((a,b),(c,d)) = k(a,c)\cdot k(b,d) + k(a,d)\cdot k(b,c)$

These are symmetric by construction because swapping $(a,b)$ to $(b,a)$ gives the same value.

**Reference:** Section 3, Equations (6)–(9) — balanced kernel definitions; Theorem 2 — formal symmetry proof.

**Purpose:** Ensures the model is invariant to input order, which is a natural requirement for identity verification tasks.

---

### Step 4: SVM Training on Pairs

**Description:**  
The SVM is trained using the standard soft-margin SVM objective, but with the pairwise kernel replacing the standard one. LIBSVM is used internally. The key insight is that the balanced pairwise kernel can be computed efficiently by caching the base kernel $k$ evaluations, so the pair-kernel evaluations don't require extra computation beyond what standard SVM training does.

**Reference:** Section 4 — efficient implementation; Theorem 4 — computational complexity analysis.

**Purpose:** Trains a maximum-margin classifier in the pairwise feature space.

---

### Step 5: Decision for a New Pair

**Description:**  
For a new test pair $(a_{test}, b_{test})$, the SVM decision function outputs:

$$f(a_{test}, b_{test}) = \text{sign}\left(\sum_i \alpha_i y_i K((a_i, b_i), (a_{test}, b_{test})) + b\right)$$

where $\alpha_i$ are support vector weights, $y_i$ are training labels, and $K$ is the balanced pairwise kernel. The output is +1 (same class) or −1 (different class).

**Reference:** Section 2, Equation (2) — general SVM decision function adapted to pairs.

**Purpose:** Classifies whether the two test inputs belong to the same identity.

---

### Final Summary

> This paper solves the **pairwise classification problem** (determining whether two inputs share the same class) by defining **symmetric pairwise kernels** for SVMs that are invariant to the order of the input pair, and it is better than naïve approaches because it mathematically guarantees symmetry while being computationally as efficient as training a standard SVM.